In [30]:
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm

from eeg.eeg_data import EEGLLM, EEGDataset
from eeg.eeg_data.datasets.utils import Colors

In [2]:
eeg_dataset: EEGDataset = EEGDataset(region_tokenizer_path="../models/appendages", print_shapes=True)

Initializing dataset...
Getting VQVAE model...
Getting EEG data...
Filtered & processed EEG data.
Getting appendage data...
Retrieved appendage data.
Successful retrieved all data.
Pre-computing VQ-VAE tokens...


100%|██████████| 25/25 [00:02<00:00,  8.47it/s]

EEG shape:           (50583, 14)
app shape:           (50583, 12)
VQ-VAE tokens shape: (50583,)
total # of chunks: 44


In [3]:
def load_model(epoch: int) -> EEGLLM:
    model = EEGLLM(
        vocab_size=512,
        num_layers=4,
        num_heads=4,
        num_channels=14,
        embedding_dim=64,
        ffn_hidden_dim=64,
        qk_length=64,
        value_length=64,
        max_length=2048,
        dropout=0.1
    )

    state_dict = torch.load(f"/var/log/thavamount/eeg_ckpts/eeg_lm/big/eeg_llm_epoch_{epoch}.pth", map_location="cpu")
    model.load_state_dict(state_dict["model"])
    
    return model

In [4]:
eeg, apps, tokens = eeg_dataset.get_val_data(0)
print(torch.Tensor(eeg).shape)
print(torch.Tensor(apps).shape)
print(torch.Tensor(tokens).shape)

torch.Size([900, 14])
torch.Size([900, 12])
torch.Size([900])


In [31]:
def inference(
    model: torch.nn.Module,
    eeg: torch.Tensor,
    tokens: torch.Tensor,
    run_num: int,
    seen_so_far: int = 100
) -> torch.Tensor:

    model.eval()

    eeg = eeg.unsqueeze(0).to(torch.int64)
    pred_tokens = [torch.tensor(i) for i in tokens[:seen_so_far].tolist()]

    print(f"{Colors.OKCYAN}Computing...{Colors.ENDC}")
    for i in tqdm(range(run_num)):
        token_logits  = model(eeg[:, :seen_so_far + i, :].to(torch.float32))

        best_token = torch.argmax(token_logits.squeeze(0)[-1])
        pred_tokens.append(best_token)

    return torch.stack(pred_tokens)  # (T, 1)

In [32]:
run_num = 200
seen_so_far = 600
model = load_model(epoch=500)
pred_tokens = inference(
    model,
    torch.Tensor(eeg),
    torch.tensor(tokens),
    run_num=run_num,
    seen_so_far=seen_so_far
)

plt.figure(figsize=(20, 5))
plt.plot(tokens[:run_num + seen_so_far], label="gt tokens")
plt.plot(pred_tokens.cpu().numpy(), label="predicted tokens")
plt.legend()
plt.title("EEG token prediction")
plt.xlabel("time steps")
plt.ylabel("token")
plt.show()

Computing...


  4%|▎         | 7/200 [00:37<17:26,  5.42s/it]


KeyboardInterrupt: 